# 08b · Recourse benchmark analysis (multi-seed)

Reads the output of `scripts/run_recourse_benchmark.py`. One row per (seed, target, gate, individual).

**What to look for**
1. **Detector transfer.** A gate built on detector A should lower `flag_A`. Does it also lower the flag rate of the *other* detectors? Compare each gate's own column with the off-diagonal columns, always against the **real favourable rows** (the right reference for counterfactuals, which live in the favourable region), not against alpha.
2. **Margin.** How do `robust`, `success` and `cost` move as `target` goes from 0.5 up?
3. **Paired evidence.** The per-method table compares different individuals (a gate can "improve" a metric just by failing on hard cases). The paired table only uses individuals where both methods found a counterfactual.

Uncertainty: the first table gives the sd across seeds (conservative). The paired CIs pool individuals over seeds and treat them as independent, so read them as optimistic.

In [1]:
import numpy as np
import pandas as pd

from conformal_trees.evaluation import paired_bootstrap_ci
from conformal_trees.utils import derive_seed, make_rng, resolve_path

pd.set_option("display.max_columns", 50)
df = pd.read_csv(resolve_path("results/raw/recourse_benchmark.csv"))
ref = pd.read_csv(resolve_path("results/raw/recourse_benchmark_reference.csv"))
METRICS = ["cost", "n_changed", "flag_IF", "flag_LOF", "flag_kNN", "robust"]
print(f"seeds {sorted(df.seed.unique())}, individuals per seed {df.groupby('seed').i.nunique().iloc[0]}, rows {len(df)}")

seeds [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)], individuals per seed 200, rows 9000


## 1. Per method: mean and sd across seeds
`success` = share of individuals with a counterfactual. The other columns are averaged over the counterfactuals that were found.
`flag_*` = share flagged by that detector's plain alpha-level conformal test. `robust` = share of re-trained models (other seeds) that still classify the counterfactual as favourable.

In [2]:
per_seed = df.groupby(["seed", "target", "gate"]).agg(
    success=("found", "mean"), **{m: (m, "mean") for m in METRICS}).reset_index()
summary = per_seed.groupby(["target", "gate"])[["success"] + METRICS].agg(["mean", "std"]).round(3)
summary

success          cost        n_changed        flag_IF         \
                   mean    std   mean    std      mean    std    mean    std   
target gate                                                                    
0.5    gate_IF    0.609  0.033  2.381  0.200     2.086  0.063   0.021  0.005   
       gate_LOF   0.633  0.031  2.414  0.182     2.122  0.058   0.087  0.020   
       none       0.638  0.031  2.417  0.189     2.103  0.060   0.089  0.024   
0.6    gate_IF    0.565  0.036  2.801  0.190     2.365  0.021   0.019  0.014   
       gate_LOF   0.594  0.042  2.866  0.182     2.407  0.043   0.087  0.021   
       none       0.597  0.044  2.861  0.210     2.381  0.040   0.093  0.022   
0.7    gate_IF    0.477  0.048  3.119  0.249     2.608  0.157   0.014  0.011   
       gate_LOF   0.494  0.056  3.168  0.252     2.630  0.153   0.068  0.025   
       none       0.506  0.061  3.207  0.281     2.635  0.157   0.082  0.036   

                flag_LOF        flag_kNN        robust         
                    mean    std     mean    std   mean    std  
target gate                                                    
0.5    gate_IF     0.069  0.040    0.064  0.010  0.546  0.051  
       gate_LOF    0.017  0.013    0.100  0.027  0.529  0.053  
       none        0.071  0.039    0.113  0.021  0.529  0.054  
0.6    gate_IF     0.088  0.025    0.077  0.024  0.907  0.026  
       gate_LOF    0.023  0.014    0.120  0.017  0.902  0.031  
       none        0.085  0.016    0.135  0.022  0.901  0.029  
0.7    gate_IF     0.075  0.035    0.062  0.039  0.988  0.009  
       gate_LOF    0.029  0.024    0.096  0.025  0.987  0.010  
       none        0.083  0.041    0.115  0.034  0.987  0.009

Reference flag rates (per detector) for the original unfavourable individuals and for real favourable test rows:

In [3]:
ref.groupby("group")[["flag_IF", "flag_LOF", "flag_kNN"]].agg(["mean", "std"]).round(3)

flag_IF        flag_LOF        flag_kNN       
                         mean    std     mean    std     mean    std
group                                                               
original_unfavourable   0.054  0.014    0.070  0.021    0.056  0.016
real_favourable         0.071  0.007    0.044  0.004    0.064  0.009

## 2. Paired comparison against `none`
Only individuals where both methods found a counterfactual. `diff` = gate minus none.

In [4]:
def paired(gate, target, metric, rng):
    key = ["seed", "i"]
    a = df[(df.gate == gate) & (df.target == target) & df.found].set_index(key)[metric]
    b = df[(df.gate == "none") & (df.target == target) & df.found].set_index(key)[metric]
    common = a.index.intersection(b.index)
    return (*paired_bootstrap_ci(a.loc[common], b.loc[common], rng=rng), len(common))


rng = make_rng(derive_seed(0, "bench", "boot"))
rows = []
for gate in sorted(set(df.gate) - {"none"}):
    for t in sorted(df.target.unique()):
        for m in METRICS:
            mean, lo, hi, n = paired(gate, t, m, rng)
            rows.append({"gate": gate, "target": t, "metric": m, "diff": mean, "ci low": lo, "ci high": hi, "n pairs": n})
paired_df = pd.DataFrame(rows)
paired_df.pivot_table(index=["gate", "target"], columns="metric", values="diff")[METRICS].round(3)

metric            cost  n_changed  flag_IF  flag_LOF  flag_kNN  robust
gate     target                                                       
gate_IF  0.5     0.028      0.023   -0.025    -0.002    -0.011   0.005
         0.6     0.025      0.023   -0.023     0.004    -0.014   0.002
         0.7     0.012      0.008   -0.015     0.000    -0.002   0.000
gate_LOF 0.5     0.024      0.032   -0.002    -0.049    -0.006  -0.002
         0.6     0.017      0.030   -0.002    -0.057    -0.010   0.001
         0.7     0.025      0.016   -0.004    -0.034     0.000   0.000

In [5]:
# Detector flags and robustness with their 95% CIs
paired_df[paired_df.metric.isin(["flag_IF", "flag_LOF", "flag_kNN", "robust"])].round(3).reset_index(drop=True)

,gate,target,metric,diff,ci low,ci high,n pairs
0,gate_IF,0.5,flag_IF,-0.025,-0.038,-0.013,609
1,gate_IF,0.5,flag_LOF,-0.002,-0.008,0.005,609
2,gate_IF,0.5,flag_kNN,-0.011,-0.021,-0.003,609
3,gate_IF,0.5,robust,0.005,-0.001,0.012,609
4,gate_IF,0.6,flag_IF,-0.023,-0.035,-0.012,565
5,gate_IF,0.6,flag_LOF,0.004,-0.004,0.011,565
6,gate_IF,0.6,flag_kNN,-0.014,-0.025,-0.004,565
7,gate_IF,0.6,robust,0.002,-0.003,0.007,565
8,gate_IF,0.7,flag_IF,-0.015,-0.025,-0.004,477
9,gate_IF,0.7,flag_LOF,0.000,-0.006,0.006,477


In [6]:
from scipy.stats import binomtest
import numpy as np
import pandas as pd

def mcnemar(gate, target, flag):
    key = ["seed", "i"]
    a = df[(df.gate == gate) & (df.target == target) & df.found].set_index(key)[flag]
    b = df[(df.gate == "none") & (df.target == target) & df.found].set_index(key)[flag]
    common = a.index.intersection(b.index)
    a, b = a.loc[common], b.loc[common]
    only_gate = int(((a == 1) & (b == 0)).sum())
    only_none = int(((a == 0) & (b == 1)).sum())
    n_disc = only_gate + only_none
    p = binomtest(only_gate, n_disc, 0.5).pvalue if n_disc else float("nan")
    return only_gate, only_none, p

def holm(pvals):
    p = np.asarray(pvals, dtype=float)
    ok = ~np.isnan(p)
    adj = np.full_like(p, np.nan)
    if not np.any(ok):
        return adj
    
    p_valid = p[ok]
    order = np.argsort(p_valid)
    m = len(p_valid)
    
    adjusted = np.empty(m, dtype=float)
    running = 0.0
    for rank, orig_idx in enumerate(order):
        val = (m - rank) * p_valid[orig_idx]
        running = max(running, val)
        adjusted[orig_idx] = min(1.0, running)
        
    adj[ok] = adjusted
    return adj

def per_seed_diffs(gate, target, metric):
    out = []
    for s in sorted(df.seed.unique()):
        d = df[df.seed == s]
        a = d[(d.gate == gate) & (d.target == target) & d.found].set_index("i")[metric]
        b = d[(d.gate == "none") & (d.target == target) & d.found].set_index("i")[metric]
        c = a.index.intersection(b.index)
        out.append(float((a.loc[c] - b.loc[c]).mean()))
    return out

OWN = {"gate_IF": "flag_IF", "gate_LOF": "flag_LOF"}
OTHERS = {
    "gate_IF": ["flag_LOF", "flag_kNN"],
    "gate_LOF": ["flag_IF", "flag_kNN"]
}

rng_rule = make_rng(derive_seed(0, "bench", "rule"))
rows = []

for gate in OWN:
    for t in sorted(df.target.unique()):
        for flag in [OWN[gate]] + OTHERS[gate]:
            mean, lo, hi, n = paired(gate, t, flag, rng_rule)
            og, on, p = mcnemar(gate, t, flag)
            seed_diffs = per_seed_diffs(gate, t, flag)
            rows.append({
                "gate": gate,
                "target": t,
                "detector": flag,
                "role": "own" if flag == OWN[gate] else "transfer",
                "diff": mean,
                "ci low": lo,
                "ci high": hi,
                "n pairs": n,
                "only_gate": og,
                "only_none": on,
                "p": p,
                "seeds with drop >= 2pts": int(sum(d <= -0.02 for d in seed_diffs)),
                "n seeds": len(seed_diffs)
            })

rule = pd.DataFrame(rows)
rule["p (Holm)"] = holm(rule["p"].to_numpy())

def verdict(r):
    if r["role"] != "transfer":
        return ""
    if r["diff"] <= -0.02 and r["ci high"] < 0:
        return "transfer"
    if abs(r["diff"]) < 0.01 and r["ci low"] <= 0 <= r["ci high"]:
        return "none"
    return "inconclusive"

rule["verdict"] = rule.apply(verdict, axis=1)
display(rule.round(4))

,gate,target,detector,role,diff,ci low,ci high,n pairs,only_gate,only_none,p,seeds with drop >= 2pts,n seeds,p (Holm),verdict
0,gate_IF,0.5,flag_IF,own,-0.0246,-0.0378,-0.0131,609,0,15,0.0001,3,5,0.0009,
1,gate_IF,0.5,flag_LOF,transfer,-0.0016,-0.0082,0.0049,609,2,3,1.0000,0,5,1.0000,none
2,gate_IF,0.5,flag_kNN,transfer,-0.0115,-0.0197,-0.0033,609,0,7,0.0156,1,5,0.2031,inconclusive
3,gate_IF,0.6,flag_IF,own,-0.0230,-0.0354,-0.0124,565,0,13,0.0002,2,5,0.0034,
4,gate_IF,0.6,flag_LOF,transfer,0.0035,-0.0018,0.0106,565,3,1,0.6250,0,5,1.0000,none
5,gate_IF,0.6,flag_kNN,transfer,-0.0142,-0.0248,-0.0035,565,1,9,0.0215,1,5,0.2363,inconclusive
6,gate_IF,0.7,flag_IF,own,-0.0147,-0.0252,-0.0042,477,0,7,0.0156,2,5,0.2031,
7,gate_IF,0.7,flag_LOF,transfer,0.0000,-0.0063,0.0063,477,1,1,1.0000,0,5,1.0000,none
8,gate_IF,0.7,flag_kNN,transfer,-0.0021,-0.0084,0.0042,477,1,2,1.0000,0,5,1.0000,none
9,gate_LOF,0.5,flag_LOF,own,-0.0490,-0.0664,-0.0332,633,0,31,0.0000,4,5,0.0000,


## 3. Notes to carry into `docs/decisions.md`
- Off-diagonal transfer: does `gate_IF` lower `flag_LOF` / `flag_kNN`, and does `gate_LOF` lower `flag_IF` / `flag_kNN`? Use the paired CIs.
- Which flag rates sit near the real-favourable reference, and which are far above or below it (over-correction)?
- Margin: smallest `target` at which `robust` is close to 1, and what it costs in `success`, `cost` and the `flag_*` rates.
- Does any conclusion change between seeds? Look at the sd in table 1 before trusting a difference.